# VALL-E — Language Model for Zero-Shot TTS

**Paper:** [Neural Codec Language Models are Zero-Shot Text to Speech Synthesizers](https://arxiv.org/abs/2301.02111) (Wang et al., Microsoft, 2023)

---

## Key Insight

VALL-E reframes TTS as a **language modeling problem on audio tokens**.

Instead of predicting mel spectrogram frames, it predicts **discrete audio codec tokens** — the same kind of tokens produced by a neural audio codec (EnCodec).

This allows VALL-E to use the **in-context learning** capability of large language models:
> *Given [text tokens] + [first 3 sec of speaker's audio tokens], predict [the rest of the audio tokens]*

No speaker encoder. No d-vector. The reference audio tokens ARE the speaker conditioning.

---

## Architecture Overview

```
Reference Audio (3 sec)
        |
   EnCodec Encoder
        |
  Reference Tokens  [r1, r2, ..., rN]   <- discrete codes
        |
        +---- [Text Tokens] ----+
                                |
                        VALL-E LM (AR + NAR)
                                |
                   Target Audio Tokens
                                |
                   EnCodec Decoder
                                |
                         Waveform
```

Two decoding stages:
- **AR (Autoregressive)**: predicts coarse (first codebook) tokens
- **NAR (Non-Autoregressive)**: predicts fine (remaining codebooks) tokens in parallel

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
print("PyTorch:", torch.__version__)

## 1. Neural Audio Codec (EnCodec)

**Paper:** [High Fidelity Neural Audio Compression](https://arxiv.org/abs/2210.13438) (Defossez et al., Meta, 2022)

EnCodec is a **neural audio codec** that compresses waveforms into discrete tokens using:
- **Encoder**: Waveform -> continuous latent (convolutional)
- **Residual Vector Quantization (RVQ)**: Continuous latent -> multiple codebooks of discrete tokens
- **Decoder**: Tokens -> Waveform

RVQ uses **8 codebooks** in series, each refining the residual error of the previous:
```
Audio -> Encoder -> z_continuous
z_continuous -> VQ1 -> code1, residual1
residual1    -> VQ2 -> code2, residual2
...
residual7    -> VQ8 -> code8

Reconstruct: code1 + code2 + ... + code8 -> Decoder -> Audio
```

For 24kHz audio with 75 frames/sec: 3 seconds = 225 time steps x 8 codebooks = **1800 tokens**.

In [ ]:
class ResidualVectorQuantizer(nn.Module):
    def __init__(self, d_latent=128, n_codebooks=8, vocab_size=1024):
        super().__init__()
        self.codebooks = nn.ModuleList([
            nn.Embedding(vocab_size, d_latent) for _ in range(n_codebooks)
        ])
        self.n_codebooks = n_codebooks
        self.vocab_size  = vocab_size

    def quantize_one(self, z, codebook):
        # z: (B, T, d_latent)
        # Find nearest codebook entry
        dist = torch.cdist(z, codebook.weight.unsqueeze(0).expand(z.size(0), -1, -1))
        idx  = dist.argmin(dim=-1)        # (B, T)
        z_q  = codebook(idx)              # (B, T, d_latent)
        return z_q, idx

    def forward(self, z):
        # z: (B, T, d_latent)
        all_codes = []
        residual  = z
        for cb in self.codebooks:
            z_q, codes = self.quantize_one(residual, cb)
            all_codes.append(codes)
            residual = residual - z_q.detach()   # compute residual
        return torch.stack(all_codes, dim=1)   # (B, n_codebooks, T)

    def decode(self, codes):
        # codes: (B, n_codebooks, T)
        z_q = sum(self.codebooks[i](codes[:, i]) for i in range(self.n_codebooks))
        return z_q   # (B, T, d_latent)

rvq = ResidualVectorQuantizer(d_latent=64, n_codebooks=4, vocab_size=512)
z = torch.randn(2, 75, 64)           # 1 second at 75 frames/sec
codes = rvq(z)                        # (2, 4, 75)
z_reconstructed = rvq.decode(codes)
print("Audio codes shape:", codes.shape)
print("  -> (batch, n_codebooks, time_steps)")
print("Reconstructed z:", z_reconstructed.shape)
print("Unique codes per codebook:", [codes[:, i].unique().numel() for i in range(4)])

## 2. VALL-E Autoregressive Model

The AR model generates the **first codebook** (coarse tokens) autoregressively.

It is a **decoder-only Transformer** (like GPT) that takes:
- Phoneme tokens (text)
- Reference audio tokens from the first codebook (speaker conditioning)
- Previously generated first-codebook tokens

And predicts the next first-codebook token.

This is where the **in-context learning** happens: the model has never seen the target speaker during training, but it can mimic the speaker by conditioning on their audio tokens.

In [ ]:
class ValleAR(nn.Module):
    def __init__(self, vocab_text=256, vocab_audio=1024, d_model=512,
                 n_heads=8, n_layers=12, max_len=2048):
        super().__init__()
        self.text_embed  = nn.Embedding(vocab_text,  d_model)
        self.audio_embed = nn.Embedding(vocab_audio + 1, d_model)   # +1 for EOS token
        self.pos_enc     = nn.Embedding(max_len, d_model)
        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4,
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerDecoder(dec_layer, num_layers=n_layers)
        self.head = nn.Linear(d_model, vocab_audio + 1)   # predict next audio token

    def forward(self, text_tokens, ref_audio_codes, target_audio_codes=None):
        # text_tokens:       (B, T_text)
        # ref_audio_codes:   (B, T_ref)   -- first codebook of reference audio
        # target_audio_codes:(B, T_target) -- first codebook of target (teacher forced)
        B = text_tokens.size(0)

        # Encode text as memory (cross-attention source)
        T_t = text_tokens.size(1)
        pos_t = torch.arange(T_t, device=text_tokens.device).unsqueeze(0)
        memory = self.text_embed(text_tokens) + self.pos_enc(pos_t)

        # Decoder input: reference tokens + target tokens (shifted right)
        if target_audio_codes is not None:
            tgt_tokens = torch.cat([ref_audio_codes, target_audio_codes], dim=1)
        else:
            tgt_tokens = ref_audio_codes
        T_tgt = tgt_tokens.size(1)
        pos_a = torch.arange(T_tgt, device=tgt_tokens.device).unsqueeze(0)
        tgt = self.audio_embed(tgt_tokens) + self.pos_enc(pos_a)

        # Causal mask for autoregressive decoding
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T_tgt, device=tgt.device)

        out = self.transformer(tgt, memory, tgt_mask=causal_mask,
                               tgt_is_causal=True)
        logits = self.head(out)   # (B, T_tgt, vocab_audio+1)
        return logits

valle_ar = ValleAR(vocab_text=100, vocab_audio=512, d_model=128, n_heads=4, n_layers=3)

# Simulate: text "Hello world", 75 ref tokens (1 sec), 150 target tokens (2 sec)
text   = torch.randint(1, 100, (2, 20))
ref_c1 = torch.randint(0, 512, (2, 75))    # reference: first codebook
tgt_c1 = torch.randint(0, 512, (2, 150))   # target: first codebook

logits = valle_ar(text, ref_c1, tgt_c1)
print("Logits shape:", logits.shape)   # (2, 75+150=225, 513)
print("AR model predicts one token at a time (coarse codebook)")
print(f"Parameters: {sum(p.numel() for p in valle_ar.parameters()):,}")

## 3. VALL-E Non-Autoregressive Model

After the AR model generates the coarse (first codebook) tokens, the **NAR model** fills in the remaining 7 codebooks in parallel.

For each codebook level `k` (from 2 to 8):
- Input: all previously generated codes (codebooks 1 to k-1) + text + reference
- Output: all `T` tokens of codebook `k` **at once** (non-autoregressive)

This makes fine-detail generation fast.

In [ ]:
class ValleNAR(nn.Module):
    def __init__(self, vocab_text=256, vocab_audio=1024, d_model=512,
                 n_heads=8, n_layers=12, n_codebooks=8):
        super().__init__()
        self.n_codebooks = n_codebooks
        # Separate embedding per codebook level
        self.audio_embeds = nn.ModuleList([
            nn.Embedding(vocab_audio, d_model) for _ in range(n_codebooks)
        ])
        self.text_embed = nn.Embedding(vocab_text, d_model)
        self.pos_enc    = nn.Embedding(4096, d_model)
        # Codebook level embedding (tells model which level to predict)
        self.level_embed = nn.Embedding(n_codebooks, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4,
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.head = nn.Linear(d_model, vocab_audio)

    def forward(self, text_tokens, all_codes, predict_level):
        # text_tokens: (B, T_text)
        # all_codes:   (B, n_prev_levels, T_audio)  -- already generated codebooks
        # predict_level: int (1 to n_codebooks-1)
        B, T_text = text_tokens.shape
        T_audio   = all_codes.size(-1)

        # Sum embeddings from all previous codebook levels
        audio_emb = sum(self.audio_embeds[k](all_codes[:, k])
                        for k in range(all_codes.size(1)))   # (B, T_audio, d_model)

        # Level conditioning: broadcast over time
        level_emb = self.level_embed(
            torch.tensor([predict_level], device=text_tokens.device)
        ).unsqueeze(1)   # (1, 1, d_model)
        audio_emb = audio_emb + level_emb

        # Text embeddings
        pos_t = torch.arange(T_text, device=text_tokens.device).unsqueeze(0)
        text_emb = self.text_embed(text_tokens) + self.pos_enc(pos_t)

        # Concatenate text + audio as flat sequence (no causal mask — NAR)
        pos_a = torch.arange(T_audio, device=text_tokens.device).unsqueeze(0)
        audio_emb = audio_emb + self.pos_enc(pos_a)
        x = torch.cat([text_emb, audio_emb], dim=1)   # (B, T_text+T_audio, d_model)

        out = self.transformer(x)
        # Predict audio tokens from audio portion only
        audio_out = out[:, T_text:, :]   # (B, T_audio, d_model)
        return self.head(audio_out)       # (B, T_audio, vocab_audio)

valle_nar = ValleNAR(vocab_text=100, vocab_audio=512, d_model=128, n_heads=4, n_layers=3, n_codebooks=4)

text       = torch.randint(1, 100, (2, 20))
prev_codes = torch.randint(0, 512, (2, 1, 100))   # first codebook already generated
logits_nar = valle_nar(text, prev_codes, predict_level=1)
print("NAR logits:", logits_nar.shape)   # (2, 100, 512)
print("NAR predicts ALL tokens of next codebook at once (parallel)")

## 4. Full VALL-E Decoding

The full inference pipeline:

```
Step 1 (AR): Generate coarse tokens autoregressively
  text + ref_c1 -> VALL-E AR -> target_c1 (T tokens, one at a time)

Step 2 (NAR): Fill in remaining codebooks in parallel
  text + ref_c1..cK + target_c1 -> VALL-E NAR -> target_c2  (all T at once)
  text + ref_c1..cK + target_c1 + target_c2 -> VALL-E NAR -> target_c3
  ...
  -> target_c8

Step 3 (Decode): EnCodec decoder
  [target_c1, target_c2, ..., target_c8] -> Waveform
```

In [ ]:
def valle_inference(valle_ar, valle_nar, text_tokens, ref_codes, max_new_tokens=150):
    valle_ar.eval()
    valle_nar.eval()
    with torch.no_grad():
        # Step 1: AR — generate first codebook autoregressively
        ref_c1 = ref_codes[:, 0, :]    # (B, T_ref) first codebook of reference
        generated = ref_c1.clone()
        for _ in range(max_new_tokens):
            logits = valle_ar(text_tokens, ref_c1, generated)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)  # greedy
            generated = torch.cat([generated, next_token], dim=1)
        target_c1 = generated[:, ref_c1.size(1):]   # strip reference prefix

        # Step 2: NAR — generate remaining codebooks
        n_cbs = valle_nar.n_codebooks
        all_target_codes = [target_c1]
        for level in range(1, n_cbs):
            prev = torch.stack(all_target_codes, dim=1)   # (B, level, T)
            logits_nar = valle_nar(text_tokens, prev, predict_level=level)
            next_codes = logits_nar.argmax(dim=-1)        # (B, T)
            all_target_codes.append(next_codes)

        target_codes = torch.stack(all_target_codes, dim=1)  # (B, n_cbs, T)
    return target_codes

# Simulate full VALL-E decoding
text       = torch.randint(1, 100, (1, 15))
ref_codes  = torch.randint(0, 512, (1, 4, 75))   # 4 codebooks, 1 sec reference

target_codes = valle_inference(valle_ar, valle_nar, text, ref_codes, max_new_tokens=30)
print("Generated audio codes:", target_codes.shape)
print("  -> (batch=1, n_codebooks=4, generated_frames=30)")
print("These would be decoded by EnCodec decoder to produce audio")

## 5. VALL-E vs SV2TTS Comparison

| Feature | SV2TTS (d-vector) | VALL-E (token-based) |
|---------|-------------------|----------------------|
| Speaker conditioning | d-vector (LSTM encoder) | Audio tokens (in-context) |
| Speaker encoder | Explicit, pre-trained | None |
| Output | Mel spectrogram | Discrete audio codes |
| Vocoder needed | Yes (HiFi-GAN) | No (EnCodec decoder) |
| In-context learning | No | Yes |
| Reference length | 3-30 sec | 3 sec |
| Language transfer | Poor | Good (large LM) |
| Training data | Hundreds of hours | 60,000+ hours (LibriLight) |
| Model size | ~100M params | ~370M params |

In [ ]:
# Visualize the two paradigms side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# SV2TTS
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 14)
ax.set_title("SV2TTS (d-vector approach)", fontsize=11, fontweight="bold")
sv2tts_boxes = [
    (3, 12, 4, 1.5, "Reference Audio\n(3-30 sec)", "#AED6F1"),
    (3,  9, 4, 1.5, "Speaker Encoder\n(LSTM, GE2E trained)", "#85C1E9"),
    (3,  6, 4, 1.5, "d-vector\n(256-dim)", "#D7BDE2"),
    (3,  3, 4, 1.5, "TTS + Conditioning\n(Tacotron2/FS2)", "#F9E79F"),
    (3,  0, 4, 1.5, "Vocoder (HiFi-GAN)", "#ABEBC6"),
]
for (x, y, w, h, label, color) in sv2tts_boxes:
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=color, edgecolor="k", lw=1.5))
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=8.5)
for y_from, y_to in [(12, 9+1.5), (9, 6+1.5), (6, 3+1.5), (3, 0+1.5)]:
    ax.annotate("", xy=(5, y_to), xytext=(5, y_from), arrowprops=dict(arrowstyle="->", lw=1.5))
ax.axis("off")

# VALL-E
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 14)
ax.set_title("VALL-E (token-based LM)", fontsize=11, fontweight="bold")
valle_boxes = [
    (0.5, 12, 3.5, 1.5, "Reference Audio\n(3 sec)", "#AED6F1"),
    (0.5,  9, 3.5, 1.5, "EnCodec Encoder\n-> Ref Tokens", "#85C1E9"),
    (6,   12, 3.5, 1.5, "Text (phonemes)", "#AED6F1"),
    (6,    9, 3.5, 1.5, "Phoneme Tokens", "#AED6F1"),
    (2,    5, 6,   2.5, "VALL-E Language Model\nAR (coarse) + NAR (fine)", "#F9E79F"),
    (2,    2, 6,   1.5, "Target Audio Tokens", "#D7BDE2"),
    (2,  0.2, 6,   1.5, "EnCodec Decoder -> Waveform", "#ABEBC6"),
]
for (x, y, w, h, label, color) in valle_boxes:
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=color, edgecolor="k", lw=1.5))
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=8.5)
ax.annotate("", xy=(2.25, 9+1.5), xytext=(2.25, 12), arrowprops=dict(arrowstyle="->"))
ax.annotate("", xy=(4, 5+2.5), xytext=(2.25, 9), arrowprops=dict(arrowstyle="->"))
ax.annotate("", xy=(7.75, 9+1.5), xytext=(7.75, 12), arrowprops=dict(arrowstyle="->"))
ax.annotate("", xy=(6, 5+2.5), xytext=(7.75, 9), arrowprops=dict(arrowstyle="->"))
ax.annotate("", xy=(5, 2+1.5), xytext=(5, 5), arrowprops=dict(arrowstyle="->"))
ax.annotate("", xy=(5, 0.2+1.5), xytext=(5, 2), arrowprops=dict(arrowstyle="->"))
ax.axis("off")

plt.suptitle("Voice Cloning: SV2TTS vs VALL-E", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figures/sv2tts_vs_valle.png", dpi=110, bbox_inches="tight")
plt.show()
print("Saved figures/sv2tts_vs_valle.png")

## Summary

| Component | Role |
|-----------|------|
| **EnCodec** | Neural audio codec — compresses waveform to discrete RVQ tokens |
| **RVQ** | 8 codebooks in series, each refining residual of previous |
| **VALL-E AR** | Autoregressive LM for first (coarse) codebook — captures speaker style |
| **VALL-E NAR** | Non-autoregressive LM for remaining (fine) codebooks |
| **In-context learning** | No speaker encoder needed — reference tokens serve as style conditioning |

**Why VALL-E is a paradigm shift:**
- TTS becomes **text completion** on audio tokens
- Any capability of large language models (few-shot, multilingual, instruction following) can in principle be applied
- Successor models (VALL-E X, VoiceCraft, Base-TTS) extend this to multilingual and longer audio generation

**Ethics note:** VALL-E requires only 3 seconds of audio to clone a voice. Microsoft added safeguards: the system includes a speaker verification step to detect if generated audio was used to impersonate real people.